# 01 — Verify Prepared RDD2022 Dataset

**Purpose:** Verify the already-prepared dataset. This notebook does NOT convert XML, copy images, or create a new split.

Expected split (do not change): train 16,636; validation 4,753; test 2,377. Total 23,766.
Classes: Longitudinal Crack, Transverse Crack, Alligator Crack, Pothole.

In [16]:
# Google Colab setup
from pathlib import Path
import os, yaml, json, shutil, random
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT = Path('/content/drive/MyDrive/RDD2022_Project')
DRIVE_DATA = DRIVE_PROJECT / 'RDD2022_Preprocessed'
LOCAL_DATA = Path('/content/RDD2022_Preprocessed')
DATA_YAML = LOCAL_DATA / 'data.yaml'
RUNS_DIR = DRIVE_PROJECT / 'YOLO_Experiments'
RUNS_DIR.mkdir(parents=True, exist_ok=True)

assert DRIVE_DATA.exists(), f'Dataset folder not found: {DRIVE_DATA}'
print('Drive dataset found:', DRIVE_DATA)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive dataset found: /content/drive/MyDrive/RDD2022_Project/RDD2022_Preprocessed


In [17]:
# Copy Drive dataset to Colab local storage for faster training.
# This is a fresh runtime copy; the Drive backup remains unchanged.
if not LOCAL_DATA.exists():
    print('Copying dataset from Drive to local Colab storage...')
    shutil.copytree(DRIVE_DATA, LOCAL_DATA)
else:
    print('Local dataset already exists; not copying again.')

print('Local dataset exists:', LOCAL_DATA.exists())

Copying dataset from Drive to local Colab storage...
Local dataset exists: True


In [20]:
# Create a local data.yaml that points to the LOCAL dataset copy.
yaml_text = '''
path: /content/RDD2022_Preprocessed
train: images/train
val: images/val
test: images/test
nc: 4
names:
  0: Longitudinal Crack
  1: Transverse Crack
  2: Alligator Crack
  3: Pothole
'''.strip() + '\n'
DATA_YAML.write_text(yaml_text)
print(DATA_YAML.read_text())

path: /content/RDD2022_Preprocessed
train: images/train
val: images/val
test: images/test
nc: 4
names:
  0: Longitudinal Crack
  1: Transverse Crack
  2: Alligator Crack
  3: Pothole



In [21]:
# Audit split counts and image-label pairing.
from collections import Counter
expected = {'train': 16636, 'val': 4753, 'test': 2377}
valid_ext = {'.jpg', '.jpeg', '.png', '.bmp'}

for split, exp_count in expected.items():
    img_dir = LOCAL_DATA / 'images' / split
    label_dir = LOCAL_DATA / 'labels' / split
    images = [p for p in img_dir.iterdir() if p.is_file() and p.suffix.lower() in valid_ext]
    labels = list(label_dir.glob('*.txt'))
    image_stems = {p.stem for p in images}
    label_stems = {p.stem for p in labels}
    missing_labels = sorted(image_stems - label_stems)
    missing_images = sorted(label_stems - image_stems)
    print(f'\\n{split.upper()}: images={len(images)}, labels={len(labels)}, expected={exp_count}')
    print('Images without labels:', len(missing_labels))
    print('Labels without images:', len(missing_images))
    assert len(images) == exp_count, f'{split} image count differs from expected'
    assert len(labels) == exp_count, f'{split} label count differs from expected'
    assert not missing_labels and not missing_images, f'{split} pairing problem'
print('\\nDATASET AUDIT PASSED')

\nTRAIN: images=16636, labels=16636, expected=16636
Images without labels: 0
Labels without images: 0
\nVAL: images=4753, labels=4753, expected=4753
Images without labels: 0
Labels without images: 0
\nTEST: images=2377, labels=2377, expected=2377
Images without labels: 0
Labels without images: 0
\nDATASET AUDIT PASSED


## Important
- Do not run a new random split here.
- Keep the test split untouched until final testing.
- If the copy cell is interrupted, rerun this notebook and inspect the counts before training.